# Training prep 

In [1]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("===Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

===Loaded preprocessing modules: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural


In [2]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("===Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

===Loaded preprocessing modules: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural


In [4]:
import pandas as pd
from pathlib import Path
import importlib

# Load df only if it is not already in memory
if "df" not in globals():
    project_root = Path.cwd().resolve().parent
    df = pd.read_csv(project_root / "data" / "train.csv")

df.head()

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### Base Cleanup
* remove: '`id`', '`LoanNr_ChkDgt`', '`Name`'
* `City` cleanup
* `Bank` cleanup
* created `IsLocalBank`
* Removed `State`
* Removed `BankState`

In [5]:
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1


###  NoEmp 
* "raw" -> en caso de árboles.( does nothing)
* "log" -> En caso de KNN o SVM for a  logarithmic transformation of NoEmp using log1p
* "binning" -> (creates bins)Para análisis interpretativo, probar en arboles también para ver si mejora o empeora el rendimiento.


* creates `NoEmp_Log` created with log, removes `NoEmp`

* creates `NoEmp_Bin` with binning, removes `NoEmp`

In [6]:
# noemp_option = "raw"   # o "log" o "binning"
noemp_option = "log"
# noemp_option = "binning"

noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option=noemp_option)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### City y Bank
binary encoding <br>
* remove `city` column
* remove `bank` column

In [7]:
city_bank = importlib.reload(city_bank)

df = city_bank.get_city_bank_encoder(df)

print("Current df features:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

df.head()



Current df features:
1. City_0
2. City_1
3. City_2
4. City_3
5. City_4
6. City_5
7. City_6
8. City_7
9. City_8
10. City_9
11. City_10
12. Bank_0
13. Bank_1
14. Bank_2
15. Bank_3
16. Bank_4
17. Bank_5
18. Bank_6
19. Bank_7
20. Bank_8
21. ApprovalDate
22. ApprovalFY
23. NewExist
24. CreateJob
25. RetainedJob
26. FranchiseCode
27. UrbanRural
28. RevLineCr
29. LowDoc
30. DisbursementDate
31. DisbursementGross
32. BalanceGross
33. Accept
34. IsLocalBank
35. NoEmp_Log


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,0,0,0,0,0,0,0,0,0,0,...,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,0,0,0,0,0,0,0,0,0,1,...,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,0,0,0,0,0,0,0,0,0,1,...,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### NewExist
* create `is_new_business`
* create `newexist_missing_or_invalid` (option A)
* remove `NewExist`

In [8]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "A" #===> is_new_business +  newexist_missing_or_invalid
# Option A: Create 'is_new_business' and 'newexist_missing_or_invalid' columns
# Option B: Create only 'is_new_business' column, and delete rows with missing/invalid values

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)


print(f"Rows after: {len(df):,}")
df.head()

NewExist option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid
0,0,0,0,0,0,0,0,0,0,0,...,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0
1,0,0,0,0,0,0,0,0,0,1,...,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0
2,0,0,0,0,0,0,0,0,0,1,...,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0
3,0,0,0,0,0,0,0,0,1,0,...,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0
4,0,0,0,0,0,0,0,0,1,0,...,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0


### CreateJob
* adds `createjob_normalized`  (option A)
* adds `createjob_standardized`  (option B)
* removes `CreateJob` column 


In [9]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "A"

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)


print(f"Rows after: {len(df):,}")
df.head()

CreateJob option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0
1,0,0,0,0,0,0,0,0,0,1,...,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0
3,0,0,0,0,0,0,0,0,1,0,...,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0
4,0,0,0,0,0,0,0,0,1,0,...,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114


### RetainedJob
* adds `retainedjob_normalized`  (option A)
* adds `retainedjob_standardized`  (option B)
* removes `RetainedJob` column 

In [10]:
# calling src\preprocessing\retainedJob.py

# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

print(f"RetainedJob option used: {retainedjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)


print(f"Rows after: {len(df):,}")
df.head()

RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0,0.0
1,0,0,0,0,0,0,0,0,0,1,...,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0,0.0
3,0,0,0,0,0,0,0,0,1,0,...,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0,0.000568
4,0,0,0,0,0,0,0,0,1,0,...,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114,0.000341


### ApprovalDate
Procesa la fecha en la que se aprobó el préstamo.
* Opción A (Recomendada para modelos geométricos/numéricos): Extrae el `ApprovalYear` (Año) y `ApprovalMonth` (Mes). Rellena los valores faltantes usando la moda (el valor más común) y aplica Normalización (MinMaxScaler) para que los valores queden en una escala de 0 a 1. Elimina la columna original.
* Opción B (Para exploración): Convierte la columna de texto original a formato datetime y la mantiene en el DataFrame `ApprovalDate` sin extraer componentes.



In [ ]:
# calling src\preprocessing\approvalDate.py
from preprocessing import approvalDate
import importlib

# Recargar el módulo para aplicar los últimos cambios en el código
approvalDate = importlib.reload(approvalDate)

# Elegir la ruta de preprocesamiento: 'A' (Extraer y Normalizar) o 'B' (Mantener datetime)
approvaldate_option = "A"

print(f"ApprovalDate option used: {approvaldate_option}")
print(f"Rows before: {len(df):,}")

# Aplicar el preprocesamiento y SOBRESCRIBIR df (para mantener el pipeline)
df = approvalDate.preprocess_approvaldate(
    df=df,
    option=approvaldate_option,
    source_col="ApprovalDate",
)

print(f"Rows after: {len(df):,}")

# Comprobación rápida de las columnas generadas
if approvaldate_option == "A":
    cols_to_show = ["approvalyear_normalized", "approvalmonth_normalized"]
    print("\nOpción A seleccionada: Se crearon las columnas normalizadas (escala 0 a 1).")
    display(df[cols_to_show].head(10))
elif approvaldate_option == "B":
    cols_to_show = ["ApprovalDate"]
    print("\nOpción B seleccionada: Se mantuvo la fecha en formato datetime.")
    display(df[cols_to_show].head(10))


Opción de ApprovalDate utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.


,ApprovalYear,ApprovalMonth
0,1996,8
1,2007,12
2,1996,5
3,2010,11
4,2007,5
5,2003,3
6,2005,10
7,1993,2
8,1999,5
9,2004,3


### ApprovalFY
Procesa el Año Fiscal del préstamo.
* Opción A: 
Removes the ApprovalFY. El equipo determinó que esta información es redundante si ya tenemos la fecha de aprobación (`ApprovalDate`).

* Opción B: Limpia la columna (elimina letras como 'A'), rellena los valores nulos con la moda, y aplica Normalización (MinMaxScaler) para dejar los años en una escala de 0 a 1<br>
creaes `approvalfy_normalized`<br>
removes `ApprovalDate`


In [ ]:
# calling src\preprocessing\approvalFY.py
from preprocessing import approvalFY
import importlib

# Recargar el módulo para aplicar los últimos cambios en el código
approvalFY = importlib.reload(approvalFY)

# Elegir la ruta de preprocesamiento: 'A' (Eliminar) o 'B' (Limpiar y Normalizar)
approvalfy_option = "A"

print(f"ApprovalFY option used: {approvalfy_option}")
print(f"Rows before: {len(df):,}")

# Aplicar el preprocesamiento y guardar en el df principal
df = approvalFY.preprocess_approvalfy(
    df=df,
    option=approvalfy_option,
    source_col="ApprovalFY",
)

print(f"Rows after: {len(df):,}")

# Comprobación rápida
if approvalfy_option == "A":
    print("\nOpción A seleccionada: La columna 'ApprovalFY' fue ELIMINADA del dataset.")
    # Mostramos que ya no está imprimiendo las primeras columnas del df
    display(df.head(3))
elif approvalfy_option == "B":
    cols_to_show = ["approvalfy_normalized"]
    print("\nOpción B seleccionada: Se creó la columna normalizada (escala 0 a 1).")
    display(df[cols_to_show].head(10))

Opción de ApprovalFY utilizada: A
Filas antes: 20,768
Filas después: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue eliminada.


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0,0.0
1,0,0,0,0,0,0,0,0,0,1,...,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0,0.0
3,0,0,0,0,0,0,0,0,1,0,...,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0,0.000568
4,0,0,0,0,0,0,0,0,1,0,...,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114,0.000341


### FranchiseCode
- 'binary': Convierte a 0 (No franquicia: códigos 0 o 1) y 1 (Sí franquicia: > 1).
- 'raw': Devuelve la columna original sin cambios (rellenando nulos con 0).

* created `IsFranchise`
* remove `FranchiseCode`
    

In [14]:
# # llamando a src\preprocessing\franchise_code.py

franchise_code = importlib.reload(franchise_code)

# Elegir la ruta de preprocesamiento de FranchiseCode: 'binary' o 'raw'
franchise_option = "binary" 

print(f"Opción de FranchiseCode utilizada: {franchise_option}")
print(f"Filas antes: {len(df):,}")

# Aplicar el preprocesamiento desde el módulo importado
df = franchise_code.preprocess_franchise_code(
    df=df,
    option=franchise_option,
    source_col="FranchiseCode",
)


print(f"Filas después: {len(df):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "FranchiseCode" in df.columns:
    cols_to_show.append("FranchiseCode")
if "IsFranchise" in df.columns:
    cols_to_show.append("IsFranchise")

display(df[cols_to_show].head(10))
df.head()


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


,IsFranchise
0,0
1,0
2,0
3,0
4,0
5,1
6,0
7,1
8,0
9,0


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,IsFranchise
0,0,0,0,0,0,0,0,0,0,0,...,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0,0.0,0
1,0,0,0,0,0,0,0,0,0,1,...,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114,0.000114,0
2,0,0,0,0,0,0,0,0,0,1,...,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0,0.0,0
3,0,0,0,0,0,0,0,0,1,0,...,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0,0.000568,0
4,0,0,0,0,0,0,0,0,1,0,...,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114,0.000341,0


### UrbanRural
- 'text': Mapea los números (0, 1, 2) a strings ('Undefined', 'Urban', 'Rural').
- 'onehot': Mapea a texto y aplica One-Hot Encoding creando 3 columnas binarias.

* created `Zone_Rural`
* created `Zone_Undefined`
* created `Zone_Urban`
* remove `UrbanRural`

In [15]:
# llamando a src\preprocessing\urban_rural.py

urban_rural = importlib.reload(urban_rural)

# Elegir la ruta de preprocesamiento de UrbanRural: 'onehot' o 'text'
urbanrural_option = "onehot" 

print(f"Opción de UrbanRural utilizada: {urbanrural_option}")
print(f"Filas antes: {len(df):,}")

# Aplicar el preprocesamiento desde el módulo importado
df = urban_rural.preprocess_urban_rural(
    df=df,
    option=urbanrural_option,
    source_col="UrbanRural",
)

print(f"Filas después: {len(df):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "UrbanRural" in df.columns:
    cols_to_show.append("UrbanRural")

# Añadir las columnas one-hot generadas si existen
zone_cols = [col for col in df.columns if col.startswith('Zone_')]
cols_to_show.extend(zone_cols)

display(df[cols_to_show].head(10))
df.head()

Opción de UrbanRural utilizada: onehot
Filas antes: 20,768
Filas después: 20,768


,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,1,0
1,0,0,1
2,0,1,0
3,0,0,1
4,0,0,1
5,0,0,1
6,0,0,1
7,0,1,0
8,0,1,0
9,0,0,1


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,0,0,0,0,0,0,0,0,0,...,1,3.367296,0,0,0.0,0.0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,1,...,1,0.693147,1,0,0.000114,0.000114,0,0,0,1
2,0,0,0,0,0,0,0,0,0,1,...,1,1.945910,1,0,0.0,0.0,0,0,1,0
3,0,0,0,0,0,0,0,0,1,0,...,1,1.791759,1,0,0.0,0.000568,0,0,0,1
4,0,0,0,0,0,0,0,0,1,0,...,1,1.386294,0,0,0.000114,0.000341,0,0,0,1


# RevLineCr

In [6]:
# calling src\preprocessing\RevLineCr.py
RevLineCr = importlib.reload(RevLineCr)

# Aplicar opción A
df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option="A",
    source_col="RevLineCr",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

# Mostrar columnas relevantes
cols_to_show = ["RevLineCr_clean"]  

rev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show += rev_cols

display(df_revlinecr[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,RevLineCr_clean,revlinecr_is_nonstandard,revlinecr_is_missing
0,N,0,0
1,N,0,0
2,N,0,0
3,N,0,0
4,N,0,0
5,UNKNOWN,1,0
6,N,0,0
7,N,0,0
8,N,0,0
9,UNKNOWN,1,0


In [8]:
# calling src\preprocessing\RevLineCr.py
RevLineCr = importlib.reload(RevLineCr)

df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option="B",
    source_col="RevLineCr",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

cols_to_show = ["RevLineCr_clean"]

ev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show += rev_cols

display(df_revlinecr[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,RevLineCr_clean
0,N
1,N
2,N
3,N
4,N
5,UNKNOWN
6,N
7,N
8,N
9,UNKNOWN


# LowDoc

In [3]:
# calling src\preprocessing\LowDoc.py
LowDoc = importlib.reload(LowDoc)

# Aplicar opción A
df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option="A",
    source_col="LowDoc",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

# Mostrar columnas relevantes
cols_to_show = ["LowDoc_clean"] 

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show += lowdoc_cols

display(df_lowdoc[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,LowDoc_clean,lowdoc_is_nonstandard,lowdoc_is_missing
0,N,0,0
1,N,0,0
2,Y,0,0
3,N,0,0
4,N,0,0
5,Y,0,0
6,N,0,0
7,N,0,0
8,N,0,0
9,N,0,0


In [5]:
# calling src\preprocessing\LowDoc.py
LowDoc = importlib.reload(LowDoc)

# Aplicar opción B
df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option="B",
    source_col="LowDoc",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

# Mostrar columnas relevantes
cols_to_show = ["LowDoc_clean"]  

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show += lowdoc_cols

display(df_lowdoc[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,LowDoc_clean
0,N
1,N
2,Y
3,N
4,N
5,Y
6,N
7,N
8,N
9,N


## DisbursementDate

Removed

In [11]:
# calling src\preprocessing\disbursementDate.py

from preprocessing import disbursementDate
import importlib

disbursementDate = importlib.reload(disbursementDate)

df_disbursementdate = disbursementDate.preprocess_disbursementdate(
    df=df,
    source_col="DisbursementDate",
)

display(df_disbursementdate.head(10))

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,2.0,0,0,1,0,N,Y,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,1.0,1,3,0,1,N,N,"$50,000.00",$0.00,0
5,4389862a9c8,6161844009,"BIG APPLE BAGELS, INC",PEORIA HEIGHTS,IL,HERITAGE BK OF CENT. ILLINOIS,IL,11-Mar-03,2003,11,2.0,0,0,10481,1,0,Y,"$118,000.00",$0.00,0
6,6f426aef103,1511985010,SUSAN ROH CORPORATION,SKOKIE,IL,CITIZENS BANK NATL ASSOC,RI,24-Oct-05,2006,3,1.0,1,4,0,1,N,N,"$50,000.00",$0.00,1
7,cb9278bd4c4,5516993007,ALPHAGRAPHICS PRINTSHOPS,BLOOMINGDALE,IL,WELLS FARGO BANK NATL ASSOC,SD,9-Feb-93,1993,5,1.0,0,0,3512,0,N,N,"$150,000.00",$0.00,1
8,2e1cef0d06a,2960204003,GARY LEE & PARTNERS,CHICAGO,IL,"SOMERCOR 504, INC.",IL,24-May-99,1999,31,1.0,6,0,1,0,N,N,"$1,000,000.00",$0.00,1
9,ca5256aeab6,7271564003,SCOTT MANUFACTURING CO INC,GRAYSLAKE (CORPORATE AND RR NA,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,16-Mar-04,2004,6,1.0,1,6,1,1,0,N,"$33,700.00",$0.00,1


## BalanceGross

Removed

In [12]:
from preprocessing import balanceGross
import importlib

balanceGross = importlib.reload(balanceGross)

df_balancegross = balanceGross.preprocess_balancegross(
    df=df,
    source_col="BalanceGross",
)

display(df_balancegross.head(10)) 

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",0
5,4389862a9c8,6161844009,"BIG APPLE BAGELS, INC",PEORIA HEIGHTS,IL,HERITAGE BK OF CENT. ILLINOIS,IL,11-Mar-03,2003,11,2.0,0,0,10481,1,0,Y,31-Oct-03,"$118,000.00",0
6,6f426aef103,1511985010,SUSAN ROH CORPORATION,SKOKIE,IL,CITIZENS BANK NATL ASSOC,RI,24-Oct-05,2006,3,1.0,1,4,0,1,N,N,30-Nov-05,"$50,000.00",1
7,cb9278bd4c4,5516993007,ALPHAGRAPHICS PRINTSHOPS,BLOOMINGDALE,IL,WELLS FARGO BANK NATL ASSOC,SD,9-Feb-93,1993,5,1.0,0,0,3512,0,N,N,31-Jul-93,"$150,000.00",1
8,2e1cef0d06a,2960204003,GARY LEE & PARTNERS,CHICAGO,IL,"SOMERCOR 504, INC.",IL,24-May-99,1999,31,1.0,6,0,1,0,N,N,17-Jan-01,"$1,000,000.00",1
9,ca5256aeab6,7271564003,SCOTT MANUFACTURING CO INC,GRAYSLAKE (CORPORATE AND RR NA,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,16-Mar-04,2004,6,1.0,1,6,1,1,0,N,31-Mar-04,"$33,700.00",1


## Accept

Converted to numeric

In [13]:
from preprocessing import accept
import importlib

accept = importlib.reload(accept)

df_accept = accept.preprocess_accept(
    df=df,
    source_col="Accept",
)

display(df_accept[["Accept"]].head(10))

,Accept
0,0
1,1
2,1
3,1
4,0
5,0
6,1
7,1
8,1
9,1


## DisbursementGross

Option A: normalize values (min-max)

In [14]:
# calling src\preprocessing\disbursementGross.py

from preprocessing import disbursementGross
import importlib

disbursementGross = importlib.reload(disbursementGross)

disbursementgross_option = "A"

df_disbursementgross = disbursementGross.preprocess_disbursementgross(
    df=df,
    option=disbursementgross_option,
    source_col="DisbursementGross",
)

print(f"DisbursementGross option used: {disbursementgross_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_disbursementgross):,}")

cols_to_show = ["DisbursementGross"]
if "disbursementgross_normalized" in df_disbursementgross.columns:
    cols_to_show.append("disbursementgross_normalized")

display(df_disbursementgross[cols_to_show].head(10))

DisbursementGross option used: A
Rows before: 20,768
Rows after: 20,768


,DisbursementGross
0,0.066704
1,0.002824
2,0.002223
3,0.008338
4,0.005559
5,0.013118
6,0.005559
7,0.016676
8,0.111173
9,0.003747


## DisbursementGross

Option B: standardize values (z-score)

In [15]:
# calling src\preprocessing\disbursementGross.py 

from preprocessing import disbursementGross
import importlib

disbursementGross = importlib.reload(disbursementGross)

disbursementgross_option = "B"

df_disbursementgross = disbursementGross.preprocess_disbursementgross(
    df=df,
    option=disbursementgross_option,
    source_col="DisbursementGross",
) 

print(f"DisbursementGross option used: {disbursementgross_option}")
print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_disbursementgross):,}")

cols_to_show = ["DisbursementGross"]
if "disbursementgross_standardized" in df_disbursementgross.columns:
    cols_to_show.append("disbursementgross_standardized")

display(df_disbursementgross[cols_to_show].head(10))

DisbursementGross option used: B
Rows before: 20,768
Rows after: 20,768


,DisbursementGross
0,1.469113
1,-0.572057
2,-0.59124
3,-0.395861
4,-0.48467
5,-0.243111
6,-0.48467
7,-0.129436
8,2.890045
9,-0.542573
